# Tome 9, Chapter 6: Homological Algebra (Python companion)

Coefficients and torsion in topological data analysis, Koszul homology and Betti numbers of monomial ideals (with a ring whose resolution depends on the characteristic), and extensions of cyclic groups by factor sets.

*Before each solution, predict what the output will look like.*

## Exercises

**P6.1 (★) CS application: which field should topological data analysis use?** Software for persistent homology (Notebook 6 of Tome 8) computes over a finite field $\mathbb F_p$, usually $\mathbb F_2$, because linear algebra over a field is fast and exact. Theorem 6.12 says what is lost. (a) Compute the integral homology of a $6$-vertex triangulation of the projective plane and of a triangulated Klein bottle with the Smith normal form, and their Betti numbers $\dim H_k(\,\cdot\,;\mathbb F_p)$ for $p=2,3$ and over $\Q$ (computed modulo a large prime). (b) Check that $\dim H_k(X;\mathbb F_p)=\operatorname{rank}H_k+t_k(p)+t_{k-1}(p)$, where $t_k(p)$ is the number of cyclic factors of the torsion of $H_k$ of order divisible by $p$, as the universal coefficient theorem predicts. (c) Which field would detect that a point cloud sampled from a Klein bottle is non-orientable?

**P6.2 (★★) Koszul homology and Betti numbers.** For a monomial ideal $I\subseteq R=K[x_1,\dots,x_n]$, the graded Betti numbers $\beta_{i,j}=\dim_K\operatorname{Tor}_i^R(R/I,K)_j$ count the generators of degree $j$ in the $i$-th module of a minimal free resolution. By the symmetry of Tor (stated without proof in the box after Proposition 6.9; see Weibel, Theorem 2.7.2), they can be computed by tensoring the Koszul resolution of $K$ with $R/I$, which is finite-dimensional linear algebra in each degree. (a) Implement this and check it on $(x,y,z)$ (Betti numbers $1,3,3,1$) and on $(x^2,xy,y^2)$. (b) Check Hilbert's syzygy theorem: $\operatorname{Tor}_i=0$ for $i>n$. (c) The *Stanley--Reisner ideal* of a simplicial complex $\Delta$ on $n$ vertices is generated by the squarefree monomials $x_S$ for the non-faces $S$. For the $6$-vertex projective plane, compute the Betti numbers in characteristic $2$ and in characteristic $3$ and compare. (Hochster's formula expresses them through the homology of subcomplexes of $\Delta$, so the torsion of $H_1(\R P^2)$ appears.)

**P6.3 (★★) Extensions by factor sets.** An abelian extension $0\to\Z/n\to E\to\Z/m\to0$ can be written on the set $\Z/n\times\Z/m$ with $(a,x)+(b,y)=(a+b+f(x,y),\,x+y)$, where $f\colon(\Z/m)^2\to\Z/n$ is a *symmetric normalised 2-cocycle*: $f(x,y)=f(y,x)$, $f(0,x)=0$ and $f(x,y)+f(x+y,z)=f(y,z)+f(x,y+z)$. Changing the set-theoretic section changes $f$ by a *coboundary* $g(x)+g(y)-g(x+y)$. (a) Enumerate all cocycles and coboundaries for small $m,n$ and check that their quotient has $\gcd(m,n)$ elements, as Example 6.8 and Theorem 6.10 predict. (b) For $m=n=3$, list the middle groups $E$ of the three classes. (Chapter 7 develops this computation for non-abelian groups.)

## Solutions

### P6.1

In [ ]:
import itertools, numpy as np
import sympy as sp
from sympy.matrices.normalforms import smith_normal_form
from sympy.polys.domains import ZZ

def closure(maximal):
    faces = set()
    for f in maximal:
        for k in range(1, len(f) + 1):
            faces.update(tuple(sorted(c)) for c in itertools.combinations(f, k))
    by_dim = {}
    for f in faces: by_dim.setdefault(len(f) - 1, []).append(f)
    return {d: sorted(v) for d, v in by_dim.items()}

def boundary(by_dim, k):
    """Matrix of d_k: C_k -> C_{k-1} (rows: (k-1)-faces, columns: k-faces)."""
    rows = {f: i for i, f in enumerate(by_dim[k - 1])}
    D = np.zeros((len(by_dim[k - 1]), len(by_dim[k])), dtype=np.int64)
    for j, f in enumerate(by_dim[k]):
        for i in range(len(f)):
            D[rows[f[:i] + f[i + 1:]], j] = (-1) ** i
    return D

def rank_mod_p(A, p):
    A = A.copy() % p; r = 0; rows, cols = A.shape
    for c in range(cols):
        piv = next((i for i in range(r, rows) if A[i, c]), None)
        if piv is None: continue
        A[[r, piv]] = A[[piv, r]]
        A[r] = A[r] * pow(int(A[r, c]), p - 2, p) % p
        for i in range(rows):
            if i != r and A[i, c]: A[i] = (A[i] - A[i, c] * A[r]) % p
        r += 1
        if r == rows: break
    return r

def integral_homology(by_dim):
    top = max(by_dim); H = {}
    D = {k: boundary(by_dim, k) for k in range(1, top + 1)}
    for k in range(top + 1):
        n_k = len(by_dim[k])
        rank_out = 0 if k == 0 else rank_mod_p(D[k], 1_000_003)
        inv = []
        if k + 1 <= top:
            S = smith_normal_form(sp.Matrix(D[k + 1].tolist()), domain=ZZ)
            inv = [abs(int(S[i, i])) for i in range(min(S.shape)) if S[i, i] != 0]
        free = n_k - rank_out - len(inv)
        H[k] = (free, [d for d in inv if d > 1])
    return H

def betti_mod_p(by_dim, p):
    top = max(by_dim); out = []
    for k in range(top + 1):
        rk_in = rank_mod_p(boundary(by_dim, k + 1), p) if k + 1 <= top else 0
        rk_out = rank_mod_p(boundary(by_dim, k), p) if k >= 1 else 0
        out.append(len(by_dim[k]) - rk_out - rk_in)
    return out

RP2 = closure([(1,2,4),(1,2,6),(1,3,5),(1,3,6),(1,4,5),(2,3,4),(2,3,5),(2,5,6),(3,4,6),(4,5,6)])
def klein(n=4):
    """n x n grid on the square with sides glued by a b a^-1 b (Figure 6.1): (i, n) ~ (i, 0), (n, j) ~ (0, n - j)."""
    def v(i, j):
        i %= 2 * n
        if i >= n: i -= n; j = n - j
        return (i % n) * n + (j % n)
    tris = []
    for i in range(n):
        for j in range(n):
            tris.append((v(i, j), v(i + 1, j), v(i + 1, j + 1))); tris.append((v(i, j), v(i, j + 1), v(i + 1, j + 1)))
    return closure(tris)
KB = klein()

for name, X in [("projective plane", RP2), ("Klein bottle", KB)]:
    H = integral_homology(X)
    print(name, " integral homology:", {k: f"Z^{f} + " + " + ".join(f"Z/{d}" for d in t) if t else f"Z^{f}" for k, (f, t) in H.items()})
    for p in (2, 3, 1_000_003):
        pred = [H[k][0] + sum(d % p == 0 for d in H[k][1]) + (sum(d % p == 0 for d in H[k - 1][1]) if k else 0) for k in H]
        print(f"   p = {p:>7}: Betti numbers {betti_mod_p(X, p)},  universal coefficients predict {pred}")

Both spaces have a $\Z/2$ in $H_1$ and nothing in $H_2$. Over $\mathbb F_2$ the torsion appears twice, in $H_1$ (through $H_1\otimes\mathbb F_2$) and in $H_2$ (through $\operatorname{Tor}_1(H_1,\mathbb F_2)$), so the projective plane has mod-$2$ Betti numbers $1,1,1$, like a space with a $2$-dimensional hole; over $\mathbb F_3$ and $\Q$ it looks like a point, $1,0,0$. For the Klein bottle, $b_2(\mathbb F_2)=1$ but $b_2(\mathbb F_3)=b_2(\Q)=0$: a closed surface has $b_2(\mathbb F_p)=1$ for odd $p$ exactly when it is orientable. So $\mathbb F_2$, the default of most software, sees a Klein bottle as having a $2$-dimensional cavity like a torus, and only a comparison with another field (or with $\Q$) detects non-orientability; running persistence over two fields is a cheap way to find torsion in data.

### P6.2

In [ ]:
from itertools import combinations, combinations_with_replacement
from collections import Counter

def monomials(n, d):
    for c in combinations_with_replacement(range(n), d):
        e = [0] * n
        for i in c: e[i] += 1
        yield tuple(e)

def in_ideal(m, gens):
    return any(all(a >= b for a, b in zip(m, g)) for g in gens)

def betti_table(n, gens, p, maxdeg):
    """beta[i][j] = dim Tor_i(R/I, K)_j over F_p, via the Koszul complex K(x_1..x_n) tensored with R/I."""
    std = {d: [m for m in monomials(n, d) if not in_ideal(m, gens)] for d in range(maxdeg + 1)}
    def basis(i, j):                      # e_S * m with |S| = i, deg m = j - i
        if j - i < 0: return []
        return [(S, m) for S in combinations(range(n), i) for m in std[j - i]]
    def matrix(i, j):                     # d: K_i -> K_{i-1} in internal degree j
        src, tgt = basis(i, j), basis(i - 1, j)
        idx = {b: k for k, b in enumerate(tgt)}
        A = np.zeros((len(tgt), len(src)), dtype=np.int64)
        for c, (S, m) in enumerate(src):
            for pos, v in enumerate(S):
                m2 = list(m); m2[v] += 1; m2 = tuple(m2)
                if in_ideal(m2, gens): continue
                A[idx[(S[:pos] + S[pos + 1:], m2)], c] += (-1) ** pos
        return A
    beta = {}
    for j in range(maxdeg + 1):
        for i in range(0, n + 1):
            dim = len(basis(i, j))
            if dim == 0: continue
            r_out = rank_mod_p(matrix(i, j), p) if i >= 1 and basis(i - 1, j) else 0
            r_in = rank_mod_p(matrix(i + 1, j), p) if i + 1 <= n and basis(i + 1, j) else 0
            b = dim - r_out - r_in
            if b: beta[(i, j)] = b
    return beta

def show(beta):
    tot = Counter()
    for (i, j), b in beta.items(): tot[i] += b
    return dict(sorted(beta.items())), [tot[i] for i in range(max(tot) + 1)]

print("(x,y,z):        ", show(betti_table(3, [(1,0,0),(0,1,0),(0,0,1)], 101, 4)))
print("(x^2,xy,y^2):   ", show(betti_table(2, [(2,0),(1,1),(0,2)], 101, 5)))

For $(x,y,z)$ the Betti numbers $1,3,3,1$ (in degrees $0,1,2,3$) are the ranks of the Koszul complex itself, which is a minimal resolution of $K$. For $(x^2,xy,y^2)$: one generator, three quadrics, and two linear syzygies in degree $3$, namely $y\cdot x^2-x\cdot xy=0$ and $y\cdot xy-x\cdot y^2=0$; nothing in homological degree $\ge3>n=2$, as Hilbert's theorem says.

In [ ]:
RP2_faces = set()
for f in [(1,2,4),(1,2,6),(1,3,5),(1,3,6),(1,4,5),(2,3,4),(2,3,5),(2,5,6),(3,4,6),(4,5,6)]:
    for k in range(1, 4):
        RP2_faces.update(combinations(sorted(f), k))
nonfaces = [S for S in combinations(range(1, 7), 3) if S not in RP2_faces]     # every pair is an edge
gens = [tuple(1 if v in S else 0 for v in range(1, 7)) for S in nonfaces]
print(len(gens), "cubic generators")
for p in (2, 3):
    print(f"char {p}:", show(betti_table(6, gens, p, 7)))

The Stanley--Reisner ideal of the $6$-vertex projective plane has $10$ cubic generators. In characteristic $3$ the Betti numbers are $1,10,15,6$, and in characteristic $2$ two extra Betti numbers appear, one in homological degree $3$ and one in degree $4$, both in internal degree $6$, the number of vertices. By Hochster's formula, $\beta_{i,6}=\dim\tilde H_{6-i-1}(\Delta;K)$ for the whole complex $\Delta$: the mod-$2$ classes in $H_1$ and $H_2$ of $\R P^2$, which exist only in characteristic $2$ (P6.1), become syzygies. So even the length of a minimal free resolution, here $3$ or $4$, depends on the characteristic of the field: the ring is Cohen--Macaulay in characteristic $3$ but not in characteristic $2$.

### P6.3

In [ ]:
from itertools import product
from math import gcd

def ext_classes(m, n):
    pairs = [(x, y) for x in range(1, m) for y in range(1, m)]
    def full(vals):
        f = {(x, y): 0 for x in range(m) for y in range(m)}
        f.update(dict(zip(pairs, vals))); return f
    cocycles = []
    for vals in product(range(n), repeat=len(pairs)):
        f = full(vals)
        if any(f[x, y] != f[y, x] for x, y in pairs): continue
        if all((f[x, y] + f[(x + y) % m, z] - f[y, z] - f[x, (y + z) % m]) % n == 0
               for x in range(m) for y in range(m) for z in range(m)):
            cocycles.append(f)
    cob = set()
    for g in product(range(n), repeat=m - 1):
        G = (0,) + g
        cob.add(tuple((G[x] + G[y] - G[(x + y) % m]) % n for x, y in pairs))
    return cocycles, cob, pairs

for m, n in [(2, 2), (2, 4), (4, 2), (3, 3), (4, 4), (2, 3)]:
    if n ** ((m - 1) ** 2) > 3 * 10**5: continue
    Z, B, pairs = ext_classes(m, n)
    print(f"m={m}, n={n}: {len(Z)} cocycles, {len(B)} coboundaries, |Ext| = {len(Z) // len(B)},  gcd = {gcd(m, n)}")

In [ ]:
def middle_group(f, m, n):
    """Invariant: the multiset of element orders of E = Z/n x Z/m with the twisted addition."""
    def add(u, v): return ((u[0] + v[0] + f[u[1], v[1]]) % n, (u[1] + v[1]) % m)
    orders = []
    for a in range(n):
        for x in range(m):
            u, k = (a, x), 1
            while u != (0, 0):
                u = add(u, (a, x)); k += 1
            orders.append(k)
    return max(orders)

Z, B, pairs = ext_classes(3, 3)
reps = {}
for f in Z:
    key = None
    for b in B:
        key = min(key or tuple(9 for _ in pairs), tuple((f[p] - c) % 3 for p, c in zip(pairs, b)))
    reps.setdefault(key, f)
for key, f in reps.items():
    e = middle_group(f, 3, 3)
    print("class", key, "-> exponent of E:", e, "->", "Z/9" if e == 9 else "Z/3 x Z/3")

The counts agree with $\gcd(m,n)$ in every case, including $(m,n)=(2,4)$ and $(4,2)$ where the two orders of the variables give the same answer even though the extensions are different ($0\to\Z/4\to E\to\Z/2\to0$ versus $0\to\Z/2\to E\to\Z/4\to0$). For $m=n=3$ there are three classes: the split one, with $E=(\Z/3)^2$, and two with $E\cong\Z/9$, inequivalent as extensions though isomorphic as groups, exactly as in Example 6.11. The cocycle $f(x,y)=\lfloor (x+y)/m\rfloor$, the *carry* of addition of digits, is the one that builds $\Z/9$ out of two copies of $\Z/3$; Chapter 7 starts from it.